# Introduction

This notebook finishes preparing the **BRK dataset** for **stock price prediction** using a **GRU model**. It combines **technical data** from Kaggle with **daily news sentiment scores**, **monthly U.S. economic recession states**, and **anomaly detection results** created in earlier steps into a single dataset.

# Methodology

## 1. Data Loading and Preparation
In this stage, we load all required datasets, including **technical market data**, **sentiment analysis outputs from the FinBERT model**, **recession classification outputs from TabPFN**, **official recession state labels**, and **TSPulse anomaly detection signals**. We prepare them for merging by **converting all date columns to a datetime format** and **removing any missing values**, although missing data is not expected at this stage.

## 2. Data Integration and Feature Selection
In this step, we **merge all datasets** to create **two final training datasets**. We first combine **technical data with sentiment features**, then merge the result separately with **official recession labels** and **TabPFN recession predictions** to produce two versions of the dataset. Finally, anomaly detection signals are added to both.
During the merging process, we **select only the required columns** from each source to keep the data clean and focused. Missing sentiment or anomaly values are replaced with simple default values to ensure the datasets are ready for model training.

## 3. Final Dataset Creation
In the final step, the **merged and cleaned datasets** are **saved as CSV files**, creating two ready-to-use datasets for **GRU model training**.

---

# Implementation

### Import Dependecies

In [1]:
import pandas as pd

### Load Technical Market Data (OHLCV and Adjusted Close)

In [2]:
# Load technical dataset
df_td = pd.read_csv("./Inputs/Technical Data/BRKS.csv")
# Verify that the dataset is loaded
df_td.head()

,date,open,high,low,close,adj close,volume
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300


### Load Sentiment Analysis Results from FinBERT Model

In [3]:
# Load sentiment analysis dataset 
df_sa = pd.read_csv("./Inputs/Sentiment Analysis/BRK_with_aggregated_sentiment_score.csv")
# Verify that the dataset is loaded
df_sa.head()

,Date,mean_sentiment,article_count,weighted_sentiment,daily_sentiment,weighted_sentiment_rolling_z_score
0,2009-10-19 00:00:00+00:00,-1.0,1,-0.693147,negative,0.000000
1,2009-11-05 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.707107
2,2009-11-08 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.577350
3,2010-01-25 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.500000
4,2010-02-22 00:00:00+00:00,-1.0,1,-0.693147,negative,1.095445


### Load Official Recession Labels (Macroeconomic Data)

In [4]:
# Load macroeconomic dataset with official recession labels
df_macro_official = pd.read_csv("./Inputs/Macroeconomics/us_macroeconomics_with_official_reccession_periods.csv")
# Verify that the dataset is loaded
df_macro_official.head()

,date,cpi,mortgage_rate,unemp_rate,nasdaq,disposable_income,personal_consumption_expenditure,personal_savings,economic_state
0,1980-11-01,85.6,14.2050,7.5,200.685556,4976.5,1826.8,11.6,Not Recession
1,1980-12-01,86.4,14.7900,7.2,198.398636,4999.8,1851.7,11.4,Not Recession
2,1981-01-01,87.2,14.9040,7.5,198.817619,4980.4,1870.0,10.9,Not Recession
3,1981-02-01,88.0,15.1325,7.4,194.852105,4965.0,1884.2,10.8,Not Recession
4,1981-03-01,88.6,15.4000,7.4,203.593182,4979.0,1902.9,10.8,Not Recession


### Load Predicted Recession Labels (TabPFN Classification)

In [5]:
# Load macroeconomic dataset with recession classification label from TabPFN
df_macro_predictions = pd.read_csv("./Inputs/Macroeconomics/us_macroeconomics_with_smote_predictions.csv")
# Verify that the dataset is loaded
df_macro_predictions.head()

,date,cpi,mortgage_rate,unemp_rate,nasdaq,disposable_income,personal_consumption_expenditure,personal_savings,economic_state
0,1980-11-01,85.6,14.2050,7.5,200.685556,4976.5,1826.8,11.6,Not Recession
1,1980-12-01,86.4,14.7900,7.2,198.398636,4999.8,1851.7,11.4,Not Recession
2,1981-01-01,87.2,14.9040,7.5,198.817619,4980.4,1870.0,10.9,Not Recession
3,1981-02-01,88.0,15.1325,7.4,194.852105,4965.0,1884.2,10.8,Not Recession
4,1981-03-01,88.6,15.4000,7.4,203.593182,4979.0,1902.9,10.8,Not Recession


### Load Anomaly Detection Results (TSPulse)

In [6]:
# Load dataset with detected anamalies
df_anomalies = pd.read_csv("./Inputs/Anomaly Detection/BRK_anomalies_processed.csv")
# Verify that the dataset is loaded
df_anomalies.head()

,date,anomaly_type,anomaly_persistent
0,2008-09-24 00:00:00+00:00,neutral,0
1,2008-09-25 00:00:00+00:00,price_dislocation,0
2,2008-09-26 00:00:00+00:00,neutral,0
3,2008-10-28 00:00:00+00:00,price_dislocation,0
4,2008-10-29 00:00:00+00:00,price_dislocation,0


---

### Merge Technical Data with Sentiment Analysis Features

In [7]:
# Drop first entry from the dataset as that entry did not have "weighted_sentiment_rolling_z_score" value.
df_sa = df_sa.drop(index=0)
# Verify that the first row was removed
df_sa.head()

,Date,mean_sentiment,article_count,weighted_sentiment,daily_sentiment,weighted_sentiment_rolling_z_score
1,2009-11-05 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.707107
2,2009-11-08 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.577350
3,2010-01-25 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.500000
4,2010-02-22 00:00:00+00:00,-1.0,1,-0.693147,negative,1.095445
5,2010-02-24 00:00:00+00:00,-1.0,1,-0.693147,negative,0.912871


In [8]:
# Convert date column of technical data DataFrame to datetime
df_td["date"] = pd.to_datetime(df_td["date"])
# Convert date column of sentiment analysis DataFrame to datetime
df_sa["Date"] = pd.to_datetime(df_sa["Date"]).dt.tz_localize(None)

In [9]:
# Create smaller sentiment analysis DataFrame that has only column that we want to use for mergning
df_sa_small = df_sa[[
    "Date",
    "daily_sentiment",
    "weighted_sentiment_rolling_z_score"
]]

In [10]:
# Merge technical data and smaller sentiment analysis DataFrame using "Date" column as merging point. 
df_td_sa = df_td.merge(
    df_sa_small,
    left_on="date",
    right_on="Date",
    how="left"
)

# Remove duplicate rows if they exist.
df_td_sa = df_td_sa.drop(columns=["Date"])

In [11]:
# Fill days that did not have news with "no_news" string
df_td_sa["daily_sentiment"] = df_td_sa["daily_sentiment"].fillna("no_news")
# Fill days that did not have "weighted_sentiment_rolling_z_score" with zeros
df_td_sa["weighted_sentiment_rolling_z_score"] = (df_td_sa["weighted_sentiment_rolling_z_score"].fillna(0))

In [12]:
# Verify merging
df_td_sa.head()

,date,open,high,low,close,adj close,volume,daily_sentiment,weighted_sentiment_rolling_z_score
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000,positive,1.140444
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300,negative,-0.438108
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100,negative,-0.343131
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800,negative,-0.566621
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300,no_news,0.000000


---

### Merge Technical & Sentiment Data with Official Recession Labels

In [13]:
# Convert date column of macroecoecnomic DataFrame with official recession labels to datetime
df_macro_official["date"] = pd.to_datetime(df_macro_official["date"])

# Create "year column" and extract year for each entry in merged technical and sentiment DataFrame
df_td_sa["year"] = df_td_sa["date"].dt.year
# Create "month" column and extract month for each entry in merged technical and sentiment DataFrame
df_td_sa["month"] = df_td_sa["date"].dt.month

# Create "year column" and extract year for each entry in macroecoecnomic DataFrame with official recession labels
df_macro_official["year"] = df_macro_official["date"].dt.year
# Create "month" column and extract month for each entry in macroecoecnomic DataFrame with official recession labels
df_macro_official["month"] = df_macro_official["date"].dt.month

# Merge technical + sentiment data with official recession labels
# The merge is done on year and month so that each day in a month gets the same recession label.
# For example, if the official data shows "Not Recession" for January 2020,
# then all daily entries in January 2020 in the technical + sentiment data will have "Not Recession".
df_td_sa_official_recessions = df_td_sa.merge(
    df_macro_official[["year", "month", "economic_state"]],
    on=["year", "month"],
    how="left"
)

# Verify merging
df_td_sa_official_recessions.head()

,date,open,high,low,close,adj close,volume,daily_sentiment,weighted_sentiment_rolling_z_score,year,month,economic_state
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000,positive,1.140444,2020,7,Not Recession
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300,negative,-0.438108,2020,7,Not Recession
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100,negative,-0.343131,2020,6,Not Recession
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800,negative,-0.566621,2020,6,Not Recession
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300,no_news,0.000000,2020,6,Not Recession


In [14]:
# Drop "year" and "month" colums from the merged dataset
df_td_sa_official_recessions = df_td_sa_official_recessions.drop(columns=["year", "month"])

In [15]:
# Verify that "year" and "month" colums were dropped
df_td_sa_official_recessions.head()

,date,open,high,low,close,adj close,volume,daily_sentiment,weighted_sentiment_rolling_z_score,economic_state
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000,positive,1.140444,Not Recession
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300,negative,-0.438108,Not Recession
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100,negative,-0.343131,Not Recession
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800,negative,-0.566621,Not Recession
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300,no_news,0.000000,Not Recession


---

### Merge Technical & Sentiment Data with Predicted Recession Labels (TabPFN)

In [16]:
# Convert date column of macroecoecnomic DataFrame with classification recession labels to datetime
df_macro_predictions["date"] = pd.to_datetime(df_macro_predictions["date"])

# Create "year column" and extract year for each entry in merged technical and sentiment DataFrame
df_td_sa["year"] = df_td_sa["date"].dt.year
# Create "month" column and extract month for each entry in merged technical and sentiment DataFrame
df_td_sa["month"] = df_td_sa["date"].dt.month

# Create "year column" and extract year for each entry in macroecoecnomic DataFrame with classification recession labels 
df_macro_predictions["year"] = df_macro_predictions["date"].dt.year
# Create "month" column and extract month for each entry in macroecoecnomic DataFrame with classification recession labels
df_macro_predictions["month"] = df_macro_predictions["date"].dt.month

# Merge technical + sentiment data with predicted recession labels
# The merge is done on year and month so that each day in a month gets the same predicted recession label.
# For example, if the predicted data shows "Not Recession" for January 2020,
# then all daily entries in January 2020 in the technical + sentiment data will have "Not Recession".
df_td_sa_predicted_recessions = df_td_sa.merge(
    df_macro_predictions[["year", "month", "economic_state"]],
    on=["year", "month"],
    how="left"
)

# Verify merging
df_td_sa_predicted_recessions.head()

,date,open,high,low,close,adj close,volume,daily_sentiment,weighted_sentiment_rolling_z_score,year,month,economic_state
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000,positive,1.140444,2020,7,Recession
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300,negative,-0.438108,2020,7,Recession
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100,negative,-0.343131,2020,6,Recession
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800,negative,-0.566621,2020,6,Recession
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300,no_news,0.000000,2020,6,Recession


In [17]:
# Drop "year" and "month" colums from the merged dataset
df_td_sa_predicted_recessions = df_td_sa_predicted_recessions.drop(columns=["year", "month"])

In [18]:
# Verify that "year" and "month" colums were dropped
df_td_sa_predicted_recessions.head()

,date,open,high,low,close,adj close,volume,daily_sentiment,weighted_sentiment_rolling_z_score,economic_state
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000,positive,1.140444,Recession
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300,negative,-0.438108,Recession
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100,negative,-0.343131,Recession
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800,negative,-0.566621,Recession
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300,no_news,0.000000,Recession


---

### Merge Technical Data, Sentiment Features, and Official Recession Labels with Anomaly Detection Results

In [19]:
# Convert date column of anamoly detection DataFrame to datetime
df_anomalies["date"] = pd.to_datetime(df_anomalies["date"]).dt.tz_localize(None)

In [20]:
# Merge technical, sentiment and macroecoecnomic DataFrame with official recession labels with anomalies DataFrame
df_td_sa_official_recessions_with_anomalies = df_td_sa_official_recessions.merge(
    df_anomalies,
    on="date",
    how="left"
)

In [21]:
# Fill days that did not have anomalies with "no_anomaly" string
df_td_sa_official_recessions_with_anomalies["anomaly_type"] = df_td_sa_official_recessions_with_anomalies["anomaly_type"].fillna("no_anomaly")

# Fill days that did not have "anomaly_persistent" with zeros
df_td_sa_official_recessions_with_anomalies["anomaly_persistent"] = df_td_sa_official_recessions_with_anomalies["anomaly_persistent"].fillna(0).astype(int)

In [22]:
df_td_sa_official_recessions_with_anomalies.head()

,date,open,high,low,close,adj close,volume,daily_sentiment,weighted_sentiment_rolling_z_score,economic_state,anomaly_type,anomaly_persistent
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000,positive,1.140444,Not Recession,neutral,0
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300,negative,-0.438108,Not Recession,neutral,0
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100,negative,-0.343131,Not Recession,neutral,0
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800,negative,-0.566621,Not Recession,neutral,0
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300,no_news,0.000000,Not Recession,neutral,0


---

### Merge Technical Data, Sentiment Features, and Predicted Recession Labels with Anomaly Detection Results

In [23]:
# Convert date column of anamoly detection DataFrame to datetime
df_anomalies["date"] = pd.to_datetime(df_anomalies["date"]).dt.tz_localize(None)

In [24]:
# Merge technical, sentiment and macroecoecnomic DataFrame with classification recession labels with anomalies DataFrame
df_td_sa_predicted_recessions_with_anomalies = df_td_sa_predicted_recessions.merge(
    df_anomalies,
    on="date",
    how="left"
)

In [25]:
# Fill days that did not have anomalies with "no_anomaly" string
df_td_sa_predicted_recessions_with_anomalies["anomaly_type"] = df_td_sa_predicted_recessions_with_anomalies["anomaly_type"].fillna("no_anomaly")

# Fill days that did not have "anomaly_persistent" with zeros
df_td_sa_predicted_recessions_with_anomalies["anomaly_persistent"] = df_td_sa_predicted_recessions_with_anomalies["anomaly_persistent"].fillna(0).astype(int)

In [26]:
df_td_sa_predicted_recessions_with_anomalies.head()

,date,open,high,low,close,adj close,volume,daily_sentiment,weighted_sentiment_rolling_z_score,economic_state,anomaly_type,anomaly_persistent
0,2020-07-02,44.020000,44.959999,43.810001,44.459999,44.459999,282000,positive,1.140444,Recession,neutral,0
1,2020-07-01,44.090000,44.200001,43.450001,43.470001,43.470001,437300,negative,-0.438108,Recession,neutral,0
2,2020-06-30,42.970001,44.439999,42.820000,44.240002,44.240002,835100,negative,-0.343131,Recession,neutral,0
3,2020-06-29,42.529999,43.369999,42.099998,42.889999,42.889999,475800,negative,-0.566621,Recession,neutral,0
4,2020-06-26,43.599998,43.630001,42.009998,42.180000,42.180000,1300300,no_news,0.000000,Recession,neutral,0


---

# Save merged DataFrames 

In [27]:
# Save technical, sentiment macroecoecnomic with classification recession labels, and anomalies DataFrame
df_td_sa_predicted_recessions_with_anomalies.to_csv(
    "./Outputs/td_sa_predicted_recessions_with_anomalies.csv",
    index=False
)

# Save technical, sentiment macroecoecnomic with official recession labels, and anomalies DataFrame
df_td_sa_official_recessions_with_anomalies.to_csv(
    "./Outputs/td_sa_official_recessions_with_anomalies.csv",
    index=False
)

print("Both files saved successfully.")

Both files saved successfully.
